


<div align="center">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apriori3d/ico/blob/main/src/examples/ml/ico_linear_regression.ipynb)

**Linear reggression example with ICO Framework**

</div>

---


### Install dependencies if running in Google Colab 

In [1]:
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("🚀 Running in Google Colab - installing dependencies...")

    # Install Apriori ICO framework from GitHub
    %pip install --upgrade -q git+https://github.com/apriori3d/ico.git

    # Install Rich for visualization and ipywidgets for progress bars
    %pip install -q rich ipywidgets

    print("✅ Dependencies installed successfully!")
else:
    print("📱 Running locally - assuming dependencies are already installed")

📱 Running locally - assuming dependencies are already installed


---

# Linear Regression with ICO Framework

This notebook demonstrates how to implement and train a simple linear regression model using the ICO framework. 

We'll show:
- **Type-safe ML pipelines** with automatic inference
- **Declarative training loops** using `IcoEpoch`
- **Rich visualization** of training progress
- **Composable operators** for data processing

The example fits a linear model `y = 2x + 1` using gradient descent, showcasing ICO's elegant approach to ML workflows.


## 📚 Table of Contents

**[1. Define the Model and Data](#1-define-the-model-and-data)**

**[2. Define Training Epoch Logic](#2-define-training-epoch-logic)**

**[3. Visualize Epoch](#3-visualize-epoch)**

**[4. Add IcoProcess for Iterations](#4-add-icoprocess-for-iterations)**

**[5. ICO Benefits Demonstrated](#5-ico-benefits-demonstrated)**

**[6. Next Steps](#6-next-steps)**




---

## 1. Define the Model and Data

First, we define our linear model as a simple dataclass with parameters and training state:

In [2]:
# Import ICO framework components
from dataclasses import dataclass
from typing import TypeAlias

from ico import source

# Type definitions for training
TrainingBatch: TypeAlias = tuple[float, float]


@dataclass
class LinearModel:
    """Simple linear regression model with training state."""

    weight: float = 0.5
    bias: float = 0.0
    total_loss: float = 0.0
    samples_processed: int = 0

    def __str__(self) -> str:
        return f"LinearModel(w={self.weight:.3f}, b={self.bias:.3f}, loss={self.total_loss:.3f})"


# Create training data source - ICO operator that generates data
@source()
def get_training_data() -> list[TrainingBatch]:
    """Generate training samples following y = 2*x + 1."""
    return [(x, 2.0 * x + 1.0) for x in range(1, 5)]

### 🔍 Inspect Data Source

The `@source()` decorator transforms our function into an ICO operator. Let's see what ICO can infer about our data.

Because we used @source decorator, get_training_data is now an ICO operator with a signature and description.

Because the provider function returns list of data samples, describe() API is able to infer total number of samples (4 in this case) and include that in the description.

In [3]:
get_training_data.describe()

───────────────────────────────────────────────── Flow plan: None ─────────────────────────────────────────────────

 Flow                                    Signature                           Name 
 📚IcoSource(get_training_data, size=4)  () → Iterator[tuple[float, float]]      

### 📋 Test Data Generation

Let's verify that our data source produces the expected training samples:

In [4]:
# Test the data source
print("Training data:")
for x, y in get_training_data(None):  # Source operator formally takes None as input
    print(f"  {x=}, {y=}")

Training data:
  x=1, y=3.0
  x=2, y=5.0
  x=3, y=7.0
  x=4, y=9.0


---

## 2. Define Training Epoch Logic

Now we implement the gradient descent update rule as a pure function:

In [5]:
def update_model(sample: TrainingBatch, model: LinearModel) -> LinearModel:
    """
    Training step that modifies the context (model) based on a data sample.

    This is the core learning logic - a pure function that takes:
    1. Input: A training sample (x, y)
    2. Context: The current model state with parameters
    3. Output: Updated model with new parameters and accumulated metrics

    In ICO terminology, this is a "context operator" that transforms the context
    (LinearModel) using information from the input (TrainingBatch).
    The updated context flows to the next sample in the epoch.
    """
    x, y = sample

    # Forward pass: prediction = weight * x + bias
    prediction = model.weight * x + model.bias
    error = prediction - y

    # Compute gradients (derivatives of loss w.r.t. parameters)
    grad_weight = error * x  # ∂Loss/∂weight = error * x
    grad_bias = error  # ∂Loss/∂bias = error

    # Update parameters using gradient descent: param -= learning_rate * gradient
    learning_rate = 0.01
    model.weight -= learning_rate * grad_weight
    model.bias -= learning_rate * grad_bias

    # Accumulate loss for monitoring (context modification for tracking)
    model.total_loss += error**2
    model.samples_processed += 1

    return model  # Modified context flows to next sample

### 🔗 Compose Training Epoch

Now we combine the data source and update logic into a complete training epoch:

Create training epoch that applies update_model to each sample from get_training_data

The IcoEpoch signature is: LinearModel → LinearModel

It takes initial model, iterates through all samples from source, applies context_operator to each sample, and returns final updated model

In [6]:
from ico import IcoEpoch

training_epoch = IcoEpoch(
    source=get_training_data,  # Where data comes from: () → Iterator[TrainingBatch]
    context_operator=update_model,  # How to update context: (TrainingBatch, LinearModel) → LinearModel
)

---

## 3. Visualize Epoch

Let's see how ICO represents our training epoch.

💡 Signature of IcoEpoch is Iterator[tuple[float, float]], LinearModel → LinearModel

It takes source of data samples with signature () → Iterator[tuple[float, float]], and apply context operator with signature (tuple[float, float], LinearModel) → LinearModel to each data sample.

LinearModel is the context that gets updated across samples in the epoch.

In [7]:
# Show the training pipeline structure
training_epoch.describe()

───────────────────────────────────────────────── Flow plan: None ─────────────────────────────────────────────────

 Flow                                        Signature                                                 Name 
 🧠IcoEpoch()                                Iterator[tuple[float, float]], LinearModel → LinearModel       
 ╭── for each in                             Iterator[tuple[float, float]]                                  
 │   📚IcoSource(get_training_data, size=4)  () → Iterator[tuple[float, float]]                             
 ├─▸ apply                                   tuple[float, float], LinearModel                               
 │   update_model                            tuple[float, float], LinearModel → LinearModel                 
 ╰─▸ emit                                    LinearModel

### Run single train epoch

In [11]:
# Initialize model with random parameters
initial_model = LinearModel(weight=0.5, bias=0.0)

# Train the model - ICO handles the iteration automatically!
trained_model = training_epoch(initial_model)

# Calculate average loss
initial_average_loss = trained_model.total_loss / trained_model.samples_processed
print(f"{initial_average_loss=:.4f}")

initial_average_loss=20.5526


---

## 4. Add IcoProcess for Iterations
We can see that IcoProcess takes an IcoEpoch as body, runs it for specified number of iterations and returns the final model after training.


In [9]:
from ico import IcoProcess

train_process = IcoProcess(
    body=training_epoch,
    num_iterations=10,  # Run for 10 iterations (epochs)
)
train_process.describe()

─────────────────────────────────────────────── Flow plan: process ────────────────────────────────────────────────

 Flow                                            Signature                                                 Name    
 ╭── iterate in 🔁IcoProcess(num_iter=10)        Iterator[tuple[float, float]], LinearModel → LinearModel  process 
 │   🧠IcoEpoch()                                Iterator[tuple[float, float]], LinearModel → LinearModel          
 │   ╭── for each in                             Iterator[tuple[float, float]]                                     
 │   │   📚IcoSource(get_training_data, size=4)  () → Iterator[tuple[float, float]]                                
 │   ├─▸ apply                                   tuple[float, float], LinearModel                                  
 │   │   update_model                            tuple[float, float], LinearModel → LinearModel                    
 │   ╰─▸ emit                                    LinearModel                                                       
 ╰─▸ emit                                        LinearModel

### Run training process

In [12]:
# Initialize model with random parameters
initial_model = LinearModel(weight=0.5, bias=0.0)
print(f"{initial_model=}")

# Train the model - ICO handles the iteration automatically!
trained_model = train_process(initial_model)

# Calculate average loss
average_loss = trained_model.total_loss / trained_model.samples_processed
print(f"{average_loss=:.4f}")

# The model should learn y = 2x + 1, so weight ≈ 2.0, bias ≈ 1.0
print("\nTarget: weight=2.0, bias=1.0")
print(f"Learned: {trained_model.weight=:.3f}, {trained_model.bias=:.3f}")

print(
    f"💡 Note: average loss is decreasing across iterations ({initial_average_loss:.4f} → {average_loss:.4f})"
    ", and learned parameters are close to target after training."
)

initial_model=LinearModel(weight=0.5, bias=0.0, total_loss=0.0, samples_processed=0)
average_loss=4.0590

Target: weight=2.0, bias=1.0
Learned: trained_model.weight=2.085, trained_model.bias=0.580
💡 Note: average loss is decreasing across iterations (20.5526 → 4.0590), and learned parameters are close to target after training.


---

## 5. ICO Benefits Demonstrated

This simple example showcases key ICO framework advantages:

### ✅ Type Safety
- Automatic type inference for the entire pipeline
- `TrainingBatch: TypeAlias = tuple[float, float]` ensures data consistency
- No runtime type errors

### ✅ Composability 
- `@source()` decorator creates reusable data operators
- `IcoEpoch` composes training logic declaratively
- Easy to extend with more complex pipelines

### ✅ Self-Describing
- `describe()` shows complete pipeline structure
- Clear signatures for debugging and understanding
- Rich console visualization

### ✅ Declarative Style
```python
# What we want, not how to do it
train_process = IcoProcess(
    body=IcoEpoch(
        source=get_training_data,
        context_operator=update_model,
    ),
    num_iterations=10,  # Run for 10 iterations (epochs)
)
```

## 6. Next Steps

### 🧠 Machine Learning
- [CIFAR-10 Classification with validation](src/examples/ml/cv/cifar/ico_cifar_complete_flow.ipynb): Complete CV pipeline replacing PyTorch DataLoader
- [CIFAR-10 Classification with worker pools](src/examples/ml/cv/cifar/ico_cifar_complete_flow_mp.py): Complete CV pipeline with parallel data processing workers
